# 11 · Monitor del indicador v3

Ejecuta desde otro kernel mientras corre el notebook 09. Solo lee el manifiesto. El porcentaje corresponde a etapas terminadas, **no a tiempo restante**: las etapas tienen costos diferentes. No elimina locks ni reanuda cálculos. La nueva versión usa DuckDB con memoria limitada; no tiene un cluster Dask activo que mostrar.


In [ ]:
from pathlib import Path
import sys,json
from IPython.display import display,Markdown
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'secop_pipeline').is_dir()),None)
if ROOT is None:raise RuntimeError('Abra el notebook dentro del proyecto.')
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from datetime import datetime
def estado_v3():
    pointer=ROOT/'salida/alertas/ejecucion_actual_v3.json'
    if not pointer.exists():
        print('Todavía no se ha iniciado una ejecución v3.');return
    work=Path(json.loads(pointer.read_text(encoding='utf-8'))['directorio'])
    r=json.loads((work/'ejecucion.json').read_text(encoding='utf-8'))
    etapas=r.get('etapas',{})
    completas=sum(v.get('estado')=='COMPLETADA' for v in etapas.values())
    display({'estado':r['estado'],'version':r['version'],'etapas_completas':completas,'total_etapas':5,
             'porcentaje_etapas':100*completas/5,'directorio':str(work),'error':r.get('error'),
             'memoria_mb':r['parametros']['memory_mb'],'hilos':r['parametros']['threads']})
    display(etapas)
    print('Locks presentes:',[str(p) for p in (ROOT/'.pipeline.lock',ROOT/'salida/alertas/.pipeline.lock') if p.exists()])
estado_v3()


Vuelve a ejecutar `estado_v3()` cuando quieras actualizar. Si figura COMPLETADA, continúa con notebooks 10, 12 o 13. Un error deja los checkpoints confirmados disponibles; no borres archivos para forzar una ejecución.